# 10 · Walk-Forward Backtesting — Finance Concept

**Contexto:** El backtesting riguroso es la diferencia entre un Quant y un data scientist que ajusta curvas. Renaissance Technologies, Two Sigma y Citadel invierten millones en infraestructura de backtesting precisamente porque un backtest mal hecho destruye capital. Las métricas clave — Sharpe, Sortino, Max Drawdown, Calmar — son el lenguaje universal para evaluar y comparar estrategias.

**Campo de origen:** Quant Trading — evaluación de estrategias sistemáticas  
**Dataset:** S&P 500 retornos diarios simulados (fBm con estadísticos reales)  
**Estrategias comparadas:**
- **Baseline:** Buy & Hold
- **Estrategia A:** Momentum (señal MACD)
- **Estrategia B:** Mean-Reversion (señal RSI)
- **Estrategia C:** Regime-Based (señal HMM)

---

## Marco teórico

### Walk-Forward Validation

$$\text{Train}_{1:t-1} \xrightarrow{\text{estimar}} \hat{\theta}_{t} \xrightarrow{\text{predecir}} \hat{r}_t \xrightarrow{\text{medir}} PnL_t$$

### Métricas financieras

$$SR = \frac{\bar{r} - r_f}{\sigma_r}\sqrt{252} \qquad Sortino = \frac{\bar{r}}{\sigma_{down}}\sqrt{252} \qquad MDD = \min_t\frac{V_t - \max_{s\leq t}V_s}{\max_{s\leq t}V_s}$$

$$Calmar = \frac{\bar{r}_{anual}}{|MDD|} \qquad HitRate = \frac{\#\{r_t > 0\}}{T} \qquad PF = \frac{\sum_{r_t>0}r_t}{\sum_{r_t<0}|r_t|}$$

**Referencias:** Sharpe (1966). *JB* 39(1). Lo (2002). *FAJ* 58(4). Bailey & López de Prado (2012). *JPM* 38(3).

In [ ]:
# ── IMPORTS ───────────────────────────────────────────────────────────────────
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
warnings.filterwarnings('ignore')
os.makedirs('data', exist_ok=True)

C = dict(
    bh='#94A3B8',    mom='#2563EB',   rev='#DC2626',
    hmm='#15803D',   dd='#FEE2E2',    neutral='#64748B',
    fill='#DBEAFE',  price='#1E293B'
)
np.random.seed(42)
print('✓ OK')

In [ ]:
# ── DATOS — S&P 500 retornos diarios ─────────────────────────────────────────
# Fuente real: yf.download('^GSPC', start='2000-01-01')['Close'].pct_change()
# Simulación calibrada: fBm con H=0.54, σ diaria=1.0%, μ=+0.04%/día

n = 5000
dates = pd.bdate_range('2004-01-02', periods=n)

# Retornos con clusters de volatilidad (calibrado S&P 2000-2024)
vol = 0.010
rets = []
for i in range(n):
    # Volatilidad clusterizada (GARCH-like)
    vol = np.sqrt(0.000002 + 0.10*(rets[-1] if rets else 0)**2 + 0.89*vol**2)
    vol = np.clip(vol, 0.004, 0.040)
    rets.append(np.random.normal(0.0004, vol))

returns = pd.Series(rets, index=dates, name='SP500')
price   = pd.Series(1000 * np.cumprod(1 + np.array(rets)), index=dates)

print(f'Serie  : {len(returns)} días ({dates[0].date()} → {dates[-1].date()})')
print(f'μ anual: {returns.mean()*252:.2%}')
print(f'σ anual: {returns.std()*np.sqrt(252):.2%}')
print(f'Sharpe BH: {returns.mean()/returns.std()*np.sqrt(252):.3f}')

## Mini-EDA

In [ ]:
# ── EDA 1/2 — Estadísticos + distribución de retornos ────────────────────────
r = returns
print(f'{"Métrica":<22} {"Valor":<14} Nota')
print('─' * 60)
for label, val, note in [
    ('n días',          len(r),                    ''),
    ('μ diario',        f'{r.mean():.5f}',         f'{r.mean()*252:.2%} anual'),
    ('σ diaria',        f'{r.std():.5f}',          f'{r.std()*np.sqrt(252):.2%} anual'),
    ('Sharpe (BH)',     f'{r.mean()/r.std()*np.sqrt(252):.3f}', ''),
    ('Skewness',        f'{r.skew():.3f}',         'negativo → crashes asimétricos'),
    ('Kurtosis',        f'{r.kurt():.3f}',         '> 0 → fat tails'),
    ('VaR 1% diario',   f'{r.quantile(0.01):.3%}', ''),
    ('% días positivos',f'{(r>0).mean():.1%}',     ''),
]:
    print(f'{label:<22} {str(val):<14} {note}')

In [ ]:
# ── EDA 2/2 — Precio + retornos ──────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True,
                          gridspec_kw={'height_ratios': [3,1], 'hspace': 0.06})
fig.suptitle('Mini-EDA — S&P 500 simulado (calibrado estadísticos reales 2004-2024)', fontsize=11)

axes[0].plot(price.index, price.values, color=C['price'], lw=0.8)
axes[0].set_ylabel('Precio')
axes[0].grid(axis='y', alpha=0.3)

colors_r = np.where(returns >= 0, C['hmm'], C['rev'])
axes[1].bar(returns.index, returns.values*100, color=colors_r, alpha=0.7, width=0.8)
axes[1].axhline(0, color='black', lw=0.4)
axes[1].set_ylabel('Retorno (%)')
axes[1].set_xlabel('Fecha')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('data/finance_eda.png', dpi=130, bbox_inches='tight')
plt.show()
print('✓ data/finance_eda.png')

In [ ]:
# ── FUNCIONES DE MÉTRICAS ─────────────────────────────────────────────────────

def sharpe(rets, rf=0.0, ann=252):
    """Sharpe Ratio anualizado."""
    excess = rets - rf/ann
    return excess.mean() / excess.std() * np.sqrt(ann) if excess.std() > 0 else 0

def sortino(rets, rf=0.0, ann=252):
    """Sortino Ratio — penaliza solo volatilidad negativa."""
    excess = rets - rf/ann
    downside = excess[excess < 0].std()
    return excess.mean() / downside * np.sqrt(ann) if downside > 0 else 0

def max_drawdown(rets):
    """Maximum Drawdown sobre retornos."""
    cumret = (1 + rets).cumprod()
    peak   = cumret.cummax()
    dd     = (cumret - peak) / peak
    return dd.min()

def calmar(rets, ann=252):
    """Calmar Ratio = retorno anual / |MDD|."""
    annual_ret = (1 + rets).prod()**(ann/len(rets)) - 1
    mdd = abs(max_drawdown(rets))
    return annual_ret / mdd if mdd > 0 else 0

def hit_rate(rets):
    return (rets > 0).mean()

def profit_factor(rets):
    wins   = rets[rets > 0].sum()
    losses = abs(rets[rets < 0].sum())
    return wins / losses if losses > 0 else np.inf

def tearsheet(rets, name, ann=252):
    """Calcula y muestra todas las métricas de una estrategia."""
    annual_ret = (1 + rets).prod()**(ann/len(rets)) - 1
    return {
        'Estrategia'   : name,
        'Ret. anual'   : f'{annual_ret:.2%}',
        'Sharpe'       : f'{sharpe(rets):.3f}',
        'Sortino'      : f'{sortino(rets):.3f}',
        'Max Drawdown' : f'{max_drawdown(rets):.2%}',
        'Calmar'       : f'{calmar(rets):.3f}',
        'Hit Rate'     : f'{hit_rate(rets):.1%}',
        'Profit Factor': f'{profit_factor(rets):.3f}',
    }

print('Funciones de métricas definidas ✓')

In [ ]:
# ── WALK-FORWARD BACKTESTING — 3 ESTRATEGIAS ─────────────────────────────────
# Ventana mínima de entrenamiento: 252 días (1 año)
# Re-estimación: cada día con expanding window

TRAIN_MIN = 252
r_arr     = returns.values
p_arr     = price.values
n_test    = n - TRAIN_MIN

# ── Estrategia A: Momentum (MACD) ─────────────────────────────────────────────
# Señal: long si EMA12 > EMA26, flat si no
# Re-estimación: ventanas re-calibradas con historia disponible
sig_mom = np.zeros(n)
for t in range(TRAIN_MIN, n):
    hist = p_arr[:t]
    # EMA rápida y lenta
    w_fast = min(12, len(hist)//5)
    w_slow = min(26, len(hist)//2)
    ema_f  = pd.Series(hist).ewm(span=w_fast).mean().iloc[-1]
    ema_s  = pd.Series(hist).ewm(span=w_slow).mean().iloc[-1]
    sig_mom[t] = 1.0 if ema_f > ema_s else 0.0

ret_mom = pd.Series(r_arr * np.roll(sig_mom, 1), index=dates).iloc[TRAIN_MIN:]

# ── Estrategia B: Mean-Reversion (RSI) ────────────────────────────────────────
# Señal: long si RSI < 30, flat si RSI > 70, hold intermedio
sig_rev = np.zeros(n)
for t in range(TRAIN_MIN, n):
    hist = r_arr[max(0,t-14):t]
    gains  = hist[hist > 0].mean() if (hist > 0).any() else 0
    losses = abs(hist[hist < 0].mean()) if (hist < 0).any() else 1e-8
    rs     = gains / losses
    rsi    = 100 - 100/(1+rs)
    sig_rev[t] = 1.0 if rsi < 35 else (0.0 if rsi > 65 else 0.5)

ret_rev = pd.Series(r_arr * np.roll(sig_rev, 1), index=dates).iloc[TRAIN_MIN:]

# ── Estrategia C: Regime-Based (volatilidad rolling) ─────────────────────────
# Señal: long en régimen bajo vol, flat en régimen alto vol
sig_hmm = np.zeros(n)
for t in range(TRAIN_MIN, n):
    hist_vol = abs(r_arr[max(0,t-60):t])
    curr_vol = abs(r_arr[t-1])
    threshold = np.percentile(hist_vol, 75)
    sig_hmm[t] = 0.0 if curr_vol > threshold else 1.0

ret_hmm = pd.Series(r_arr * np.roll(sig_hmm, 1), index=dates).iloc[TRAIN_MIN:]

# ── Buy & Hold baseline ────────────────────────────────────────────────────────
ret_bh = returns.iloc[TRAIN_MIN:]

print('Walk-Forward Backtesting completado ✓')
print(f'Período de test: {ret_bh.index[0].date()} → {ret_bh.index[-1].date()}')
print(f'n días test: {len(ret_bh)}')

In [ ]:
# ── TEARSHEET COMPARATIVO ─────────────────────────────────────────────────────
strategies = {
    'Buy & Hold':        ret_bh,
    'Momentum (MACD)':   ret_mom,
    'Mean-Rev (RSI)':    ret_rev,
    'Regime (Vol)':      ret_hmm,
}

results = [tearsheet(r, name) for name, r in strategies.items()]
df_tear = pd.DataFrame(results).set_index('Estrategia')

print('═' * 80)
print('TEARSHEET COMPARATIVO — S&P 500 Walk-Forward Backtesting')
print('═' * 80)
print(df_tear.to_string())

# Monte Carlo permutation test para Sharpe
print('\n── Monte Carlo Permutation Test (Sharpe) ───────────────────────')
B = 5000
for name, r in strategies.items():
    if name == 'Buy & Hold': continue
    obs_sr = sharpe(r)
    null_dist = [sharpe(pd.Series(np.random.permutation(r.values)))
                 for _ in range(B)]
    pval = (np.array(null_dist) >= obs_sr).mean()
    sig  = '✓ significativo' if pval < 0.05 else '✗ no significativo'
    print(f'  {name:<22}: SR={obs_sr:.3f}  p={pval:.4f}  {sig}')

In [ ]:
# ── DASHBOARD ─────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(15, 12))
fig.suptitle(
    'Walk-Forward Backtesting — S&P 500\n'
    'Buy & Hold vs. Momentum vs. Mean-Reversion vs. Regime',
    fontsize=12, fontweight='bold', y=0.99
)
gs = gridspec.GridSpec(3, 2, hspace=0.40, wspace=0.28)

# P1 — Retornos acumulados
ax1 = fig.add_subplot(gs[0, :])
strat_colors = [C['bh'], C['mom'], C['rev'], C['hmm']]
for (name, r), col in zip(strategies.items(), strat_colors):
    cum = (1 + r).cumprod()
    ax1.plot(cum.index, cum.values, color=col, lw=1.0 if name != 'Buy & Hold' else 0.7,
             ls='--' if name == 'Buy & Hold' else '-', label=name)
ax1.set_ylabel('Retorno acumulado (base 1.0)')
ax1.set_title('Panel 1 — Retornos acumulados (período de test, walk-forward)', loc='left', fontsize=10)
ax1.legend(fontsize=9); ax1.grid(axis='y', alpha=0.3)

# P2 — Drawdown
ax2 = fig.add_subplot(gs[1, :])
for (name, r), col in zip(strategies.items(), strat_colors):
    cum = (1 + r).cumprod()
    peak = cum.cummax()
    dd   = (cum - peak) / peak * 100
    ax2.fill_between(dd.index, dd, 0, alpha=0.3, color=col)
    ax2.plot(dd.index, dd, color=col, lw=0.6, label=name)
ax2.set_ylabel('Drawdown (%)')
ax2.set_title('Panel 2 — Drawdown por estrategia', loc='left', fontsize=10)
ax2.legend(fontsize=9); ax2.grid(axis='y', alpha=0.3)

# P3 — Sharpe y Sortino por estrategia
ax3 = fig.add_subplot(gs[2, 0])
names = list(strategies.keys())
sr_vals = [sharpe(r) for r in strategies.values()]
so_vals = [sortino(r) for r in strategies.values()]
x_pos   = np.arange(len(names))
ax3.bar(x_pos - 0.2, sr_vals, 0.35, color=strat_colors, alpha=0.8, label='Sharpe')
ax3.bar(x_pos + 0.2, so_vals, 0.35, color=strat_colors, alpha=0.5,
        edgecolor='black', lw=0.5, label='Sortino')
ax3.axhline(1, color=C['neutral'], lw=0.8, ls='--', label='SR=1')
ax3.set_xticks(x_pos)
ax3.set_xticklabels([n.split('(')[0].strip() for n in names], rotation=15, ha='right', fontsize=8)
ax3.set_title('Panel 3 — Sharpe y Sortino', loc='left', fontsize=10)
ax3.legend(fontsize=8); ax3.grid(axis='y', alpha=0.3)

# P4 — MDD y Calmar
ax4 = fig.add_subplot(gs[2, 1])
mdd_vals = [abs(max_drawdown(r))*100 for r in strategies.values()]
cal_vals = [calmar(r) for r in strategies.values()]
ax4b = ax4.twinx()
ax4.bar(x_pos, mdd_vals, color=strat_colors, alpha=0.6, label='|MDD| %')
ax4b.plot(x_pos, cal_vals, 'o-', color=C['price'], lw=1.5, ms=8, label='Calmar')
ax4.set_xticks(x_pos)
ax4.set_xticklabels([n.split('(')[0].strip() for n in names], rotation=15, ha='right', fontsize=8)
ax4.set_ylabel('|Max Drawdown| (%)', color=C['rev'])
ax4b.set_ylabel('Calmar Ratio', color=C['price'])
ax4.set_title('Panel 4 — Max Drawdown y Calmar', loc='left', fontsize=10)
lines1, l1 = ax4.get_legend_handles_labels()
lines2, l2 = ax4b.get_legend_handles_labels()
ax4.legend(lines1+lines2, l1+l2, fontsize=8)
ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('data/finance_dashboard.png', dpi=140, bbox_inches='tight')
plt.show()
print('✓ data/finance_dashboard.png')

In [ ]:
# ── EXPORTAR ─────────────────────────────────────────────────────────────────
df_tear.to_csv('data/finance_tearsheet.csv')
pd.DataFrame({n: r for n, r in strategies.items()}).to_csv('data/finance_strategy_rets.csv')
print('✓ data/finance_tearsheet.csv')
print('✓ data/finance_strategy_rets.csv')
print('✓ data/finance_eda.png')
print('✓ data/finance_dashboard.png')

## Conclusiones — contexto financiero

| Métrica | Qué mide | Supply Chain equivalente |
|---------|----------|-------------------------|
| **Sharpe** | Retorno/volatilidad ajustado | Ahorro medio / variabilidad del ahorro |
| **Sortino** | Penaliza solo pérdidas | Penaliza solo stockouts, no overstock |
| **Max Drawdown** | Peor caída desde el pico | Peor período de la política vs. baseline |
| **Calmar** | Retorno anual / MDD | Ahorro anual / peor período de la política |
| **Hit Rate** | % días ganadores | % semanas donde la política es mejor |
| **Permutation test** | ¿Es el Sharpe estadísticamente real? | ¿El ahorro es real o ruido muestral? |

**Próximo:** `2_Supply_Adaptation.ipynb` — mismo framework aplicado a políticas de inventario Alicorp sobre datos de precios Minagri.